## **BronzeWork Incremental**

### Step 1 - Import and Setup

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime, UTC
import uuid

In [0]:
spark.sql("use catalog novacart_adb")

In [0]:
spark.sql("create schema if not exists bronze_schema")

### Step 2 - Bronze Control Table
This table stores the **watermark** for each source table

It helps pipeline remember:
- The latest timestamp already processed
- The latest primary key processed at that timestamp
- How many rows written in the laytest run

This makes the Bronze layer load incremental and re-run safe

In [0]:
spark.sql("""
          create table if not exists novacart_adb.bronze_schema.ingestion_control(
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
          )
          using delta
          """)

### Step 3 - Source Table Configuration
This cell defines which table will be loaded into Bronze and which column should be used as :
- Primary Key
- timestamp / watermark column

It also creates a ubique bronze_run_id for the current pipeline run

In [0]:
tables_config = {
    "orders" : {"pk_col" : "order_id", "ts_col" : "updated_at"},
    "products" : {"pk_col" : "product_id", "ts_col" : "updated_at"},
    "payments" : {"pk_col" : "payment_id", "ts_col" : "processed_at"}
}

bronze_run_id = str(uuid.uuid4())

print("current bronze run id: ",bronze_run_id)

### Step - 4 Helper functions
This cell contains the reusable functions:
- **get_last_successful_watermark()** reads the last processed watermark from the control table
- **upsert_bronze_control()** updates the control table after successful Bronze load

These functions keep the main logic cleaner and easier to understand

In [0]:
def get_last_successful_watermark(tablename:str):
    ctrl = (spark.table("novacart_adb.bronze_schema.ingestion_control")
            .filter(
                (col("layer")=="bronze") & 
                (col("table_name") == tablename) & 
                (col("run_status") == "success")
            )
            .orderBy(col("updated_at").desc())
            .limit(1)
    )
    rows = ctrl.collect()
    if not rows:
        return None, None
    return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]


In [0]:
def upsert_bronze_ctrl(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):
    ctrl_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.now(UTC)
        )],
            schema = """
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
            """
    )

    dt = DeltaTable.forName(spark, "novacart_adb.bronze_schema.ingestion_control")
    dt.alias("trg").merge(ctrl_df.alias("src"), "trg.table_name = src.table_name and trg.layer = src.layer")\
        .whenMatchedUpdate(set={
            "ts_col" : "src.ts_col",
            "pk_col" : "src.pk_col",
            "last_successful_ts" : "src.last_successful_ts",
            "last_successful_pk" : "src.last_successful_pk",
            "last_run_id" : "src.last_run_id",
            "rows_written" : "src.rows_written",
            "run_status" : "src.run_status",
            "updated_at" : "src.updated_at"
        })\
        .whenNotMatchedInsertAll()\
        .execute()

### Step 5 - Bronze Incremental Load Loop

In [0]:
for table_name, cfg in tables_config.items():
    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]
    source_table = f"`novacart_sql_connection_catalog`.dbo.{table_name}"
    target_table = f"novacart_adb.bronze_schema.{table_name}_raw"
    last_successful_ts,last_successful_pk = get_last_successful_watermark(table_name)
    print(f"\n====== Processing for {target_table} ======")
    print(f"last successful ts: {last_successful_ts}")
    print(f"last successful pk: {last_successful_pk}")

    source_df = spark.read.table(source_table).withColumn(ts_col, col(ts_col).cast("timestamp"))
    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (col(ts_col) > lit(last_successful_ts)) |
            ((col(ts_col) == lit(last_successful_ts)) & (col(pk_col).cast("long") > lit(last_successful_pk)))
        )

    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("bronze_run_id", lit(bronze_run_id))
        .withColumn("bronzre_source_table", lit(source_table))
    )
    row_count = rows_to_load.count()
    print(f"{table_name} rows_to_load = {row_count}")

    if row_count == 0:
        print(f"No new rows for - {target_table}")
        upsert_bronze_ctrl(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            row_count,
            bronze_run_id
        )
        continue
    else:
        rows_to_load.write.format('delta').mode('append').saveAsTable(target_table)

        max_ts = rows_to_load.agg(max(ts_col).alias('max_ts')).collect()[0]['max_ts']
        max_pk = rows_to_load.filter(col(ts_col) == lit(max_ts)).agg(max(col(pk_col)).cast("long").alias('max_pk')).collect()[0]['max_pk']

        upsert_bronze_ctrl(
            table_name,
            ts_col,
            pk_col,
            max_ts,
            max_pk,
            row_count,
            bronze_run_id
        )

        print(f"Wrote {row_count} rows to {target_table}")



In [0]:
print("Orders Bronze Count: ", spark.table("novacart_adb.bronze_schema.orders_raw").count())
print("Products Bronze Count: ", spark.table("novacart_adb.bronze_schema.products_raw").count())
print("Payments Bronze Count: ", spark.table("novacart_adb.bronze_schema.payments_raw").count())
display(spark.sql("select * from novacart_adb.bronze_schema.ingestion_control").orderBy(col("updated_at").desc()))

In [0]:
%sql
select * from novacart_adb.bronze_schema.products_raw